In [ ]:
import sys
path = '/Users/xiehao/Desktop/workspace/X2AI/FinAI/'
if path not in sys.path:
    sys.path.append(path)

from Base import DBFile
import pandas as pd
from pyecharts.charts import Bar
from pyecharts import options as opts


pd.options.mode.chained_assignment = None  # 完全关闭警告
pd.set_option('display.float_format', '{:.3f}'.format)  # 控制所有浮点数显示格式

start_date = "2025-01-01"
end_date = "2025-06-24"

bar1d = DBFile().read_dataframe(table="future_bar1d", filters={"date": [start_date, end_date]})
reports = DBFile().read_dataframe(table="aibot_sentimental_sugar_researcher", filters={"date": [start_date, end_date]})
# 剔除置信度低的分析报告
reports = reports[reports["confidence"]>0.5]

# TEMPLET = """
# # 第 {i} 篇
# ### 发布日期: {date}
# ### 文章标题: {title}
# ### 分类: {category}
# ### 子分类: {sub_category}
# ### 短期预测: {short_forecast}
# ### 长期预测: {long_forecast}
# ### 置信区间: {confidence}å
# ### AI总结:

# {summary}

# ### AI观点:

# {opinion}
# as da s da s
# """

# result = ""
# for i in range(0, df.shape[0]):
#     row = df.iloc[i, :]
#     a = TEMPLET.format(
#         i=i,
#         date=row["date"],
#         title=row["title"],
#         category=row["category"],
#         sub_category=row["sub_category"],
#         summary=row["summary"],
#         opinion=row["opinion"],
#         short_forecast=row["short_forecast"],
#         long_forecast=row["long_forecast"],
#         confidence=row["confidence"]
#     )
#     result += a

# with open("a.md", 'w', encoding='utf-8') as file:
#     file.write(result)


# plt_df = pd.merge(rank, bar1d[["date", "instrument", "close", "ret_5", "ret_22"]], how="left", on=["date"])
# plt_df = plt_df.dropna(subset=["instrument"])
# plt_df[["short_rank", "long_rank"]] = plt_df[["short_rank", "long_rank"]].round(5)

# bar = (
#     Bar()
#     .add_xaxis(plt_df["date"].astype(str).tolist())
#     .add_yaxis("示例数据", plt_df["short_rank"].tolist(), label_opts=opts.LabelOpts(is_show=False))
#     .set_global_opts(
#         title_opts=opts.TitleOpts(title="示例柱状图"),
#         xaxis_opts=opts.AxisOpts(name="类别"),
#         yaxis_opts=opts.AxisOpts(name="数值"),
#         toolbox_opts=opts.ToolboxOpts(),
#         datazoom_opts=[opts.DataZoomOpts()]
#     )
# )
# bar.render_notebook()


In [ ]:
# 简单统计
df = reports[['date', 'title', 'short_forecast', 'long_forecast', 'confidence']].dropna()

def simple_static(today: str, df: pd.DataFrame) -> pd.DataFrame:
    """简单统计平均数、中位数、分位数"""
    stats = df[['short_forecast', 'long_forecast', 'confidence']].agg(
        ['mean', 'median', lambda x: x.quantile(0.3), lambda x: x.quantile(0.5), lambda x: x.quantile(0.8)]
    ).T
    stats.columns = ['mean', 'median', 'quantile_30', 'quantile_50', 'quantile_80']
    stats = stats.reset_index().rename(columns={'index': 'indicator'})
    stats['date'] = today
    return stats

static_df = (
    df.groupby("date")
    .apply(lambda g: simple_static(g.name, g))
    .reset_index(drop=True)
)
static_df

In [ ]:
# 分组统计
def group_static(today: str, df: pd.DataFrame) -> pd.DataFrame:
    """分组统计，包含置信度均值"""
    def _group_static(series: pd.Series, conf: pd.Series) -> dict:
        bullish = series > 0
        bearish = series < 0
        total = len(series)
        return {
            'bullish_count': bullish.sum(),
            'bearish_count': bearish.sum(),
            'bullish_ratio': bullish.sum() / total if total else None,
            'bearish_ratio': bearish.sum() / total if total else None,
            'bullish_mean': series[bullish].mean(),
            'bearish_mean': series[bearish].mean(),
            'bullish_conf_mean': conf[bullish].mean(),
            'bearish_conf_mean': conf[bearish].mean()
        }
    records = []
    for field in ['short_forecast', 'long_forecast']:
        stats = _group_static(df[field], df['confidence'])
        stats['type'] = field
        records.append(stats)
    result = pd.DataFrame(records)
    result["date"] = today
    return result

static_df = (
    df.groupby("date")
    .apply(lambda g: group_static(g.name, g))
    .reset_index(drop=True)
)
static_df

In [ ]:
# 简单加权平均法
df = reports[['date', 'short_forecast', 'long_forecast', 'confidence']].dropna()
df["short_forecast_weighted"] = df["short_forecast"] * df["confidence"]
df["long_forecast_weighted"] = df["long_forecast"] * df["confidence"]
weighted_forecast = df.groupby("date", as_index=False)[["short_forecast_weighted", "long_forecast_weighted"]].mean()
weighted_forecast

In [43]:

pd.options.mode.chained_assignment = None  # 完全关闭警告
df = weighted_forecast.copy()
df = df.sort_values('date')
decay_rate = 0.5
result = {}
for i in range(5, df.shape[0]):
    recent = df.iloc[i-5: i, :]
    max_date = recent['date'].max()
    recent['days_passed'] = (max_date - recent['date']).dt.days
    # 计算权重
    recent['weight'] = np.exp(-decay_rate * recent['days_passed'])
    # 归一化权重
    recent['weight'] = recent['weight'] / recent['weight'].sum()
    result[max_date] = {
        "short_forecast_timeweighted": (recent["short_forecast_weighted"] * recent["weight"]).sum(),
        "long_forecast_timeweighted": (recent["long_forecast_weighted"] * recent["weight"]).sum()
    }
plt_df = pd.DataFrame(result).T.reset_index().rename(columns={"index": "date"})

bar = (
    Bar()
    .add_xaxis(plt_df["date"].astype(str).tolist())
    .add_yaxis("示例数据", plt_df["short_forecast_timeweighted"].tolist(), label_opts=opts.LabelOpts(is_show=False))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="示例柱状图"),
        xaxis_opts=opts.AxisOpts(name="类别"),
        yaxis_opts=opts.AxisOpts(name="数值"),
        toolbox_opts=opts.ToolboxOpts(),
        datazoom_opts=[opts.DataZoomOpts()]
    )
)
bar.render_notebook()

In [ ]:
# 回测调整

In [ ]:
# 主题聚类分析